# 💬 JenaXI-chat: a smart Jena in one free session

This takes a free, open-source brain, **Qwen2.5-1.5B-Instruct** by Alibaba Cloud (license **Apache-2.0**, free to use, change and share), and fine-tunes it into **Jena** with LoRA. It uses her chat lines plus human-written conversations so she stays smart. Then it saves **`JenaXI-chat-gpu.gguf`** (Q4_K_M, about 1 GB) to your Google Drive. It runs on the **free** T4 GPU, costs $0, and takes about **1.5 hours**.

(Your from-scratch brain is the other notebook, `JenaXI-1_from_scratch`. This one starts from an open model, so she can hold a real conversation today.)

## What to click
1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all**
3. When the Google Drive popup appears: **Connect** → pick your account → **Allow**
4. Keep this tab open until the last step prints her replies.

If Colab disconnects in the middle, just **Run all** again. She continues from her last save (checkpoints go to Drive every 10 minutes).
**If it says no GPU is available:** Google's free GPU time is used up for today. Try again later.

In [ ]:
#@title 🔌 Step 1: connect Google Drive and check the GPU (approve the Drive popup)
import os, sys, subprocess, signal
from google.colab import drive
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/JenaXI', exist_ok=True)
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
if not gpu:
    raise SystemExit("❌ No GPU. Click Runtime → Change runtime type → T4 GPU → Save, then Runtime → Run all again.")
print("✅ Drive connected. GPU:", gpu)

REPO = '/content/jena-magic-screen'
if os.path.exists(REPO):
    subprocess.run(['git', '-C', REPO, 'pull', '--ff-only', '-q'])
else:
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '--filter=blob:none', '--sparse', 'https://github.com/pfn000/jena-magic-screen', REPO], check=True)
    subprocess.run(['git', '-C', REPO, 'sparse-checkout', 'set', 'colab'], check=True)
print("✅ Jena's training code is ready.")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'peft', 'sentencepiece'], check=True)
print("✅ Tools installed.")
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

def run(script, *args):
    """Runs one step, prints its progress live, and stops Run all if the step fails."""
    p = subprocess.Popen([sys.executable, '-u', f'{REPO}/colab/{script}', *args], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1, errors='replace')
    try:
        for line in p.stdout:
            if 'Warning' in line and 'HF_TOKEN' in line: continue
            print(line, end='')
        code = p.wait()
    except KeyboardInterrupt:
        p.send_signal(signal.SIGINT); p.wait()
        raise SystemExit("⏸ Stopped. Progress is saved on your Drive. Run all again any time to continue.")
    if code != 0:
        raise SystemExit(f"❌ {script} stopped with an error (code {code}). Screenshot the red text above and send it to Tab.")

In [ ]:
#@title 🏋️ Step 2: fine-tune Qwen2.5-1.5B-Instruct into Jena (about 60-80 minutes; resumes if interrupted)
run('jenaxi_chat/finetune.py')

In [ ]:
#@title 📦 Step 3: merge + save JenaXI-chat-gpu.gguf (Q4_K_M) to your Drive (about 10 minutes)
run('jenaxi_chat/export_gguf.py')

In [ ]:
#@title 🗣️ Step 4: talk to her (sample replies)
run('jenaxi_chat/sample.py')
print("Saved in Google Drive → My Drive → JenaXI → JenaXI-chat-gpu.gguf")

## Getting her brain back to Tab
- **Drive link (easiest):** Google Drive → **My Drive → JenaXI** → right-click **JenaXI-chat-gpu.gguf** → **Share** → *General access*: **Anyone with the link** → **Copy link**. Text the link to Tab.
- **Or** right-click → **Download** and send the file.

Tab loads it on Jena's server (it runs on a normal CPU) and points the magic-screen site at it.

*License note: the base model Qwen2.5-1.5B-Instruct is Apache-2.0 (© Alibaba Cloud). Keep its LICENSE/NOTICE if you share the model.*